# Binary Treatment Causal Analysis


In [ ]:
import warnings

import numpy as np
import pandas as pd
from dowhy import CausalModel
from IPython.display import display
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr
from sklearn.exceptions import DataConversionWarning
from sklearn.preprocessing import RobustScaler
import graphviz
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=DataConversionWarning)

import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "utils" / "statistical_tests.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing utils/statistical_tests.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import statistical_tests as tests


In [ ]:
INPUT_FILE = "alerts_with_lib_category_and_apk_size_semgrep.csv"


def normalize_verdict(value):
    normalized = str(value).strip().lower()
    if normalized in {"1", "true"}:
        return 1
    if normalized in {"0", "false"}:
        return 0
    raise ValueError(f"Unsupported verdict value: {value!r}")


df = pd.read_csv(INPUT_FILE)
print("Original rows:", len(df))

df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
print("After removing obfuscated:", len(df))

df["verdict"] = df["verdict"].apply(normalize_verdict)
df["is_third_party"] = (~df["code_location"].isin(["developer_written"])).astype(int)
print(df["is_third_party"].value_counts())

df["lib_category"] = df["code_location"].apply(lambda x: x if x != "developer_written" else "source")

# Combined approach: treated rows are all non-obfuscated alerts; control rows are developer-written alerts.
df_all = df.copy()
df_all["combined"] = 1
df_developer_written = df[df["code_location"] == "developer_written"].copy()
df_developer_written["combined"] = 0
df = pd.concat([df_all, df_developer_written], ignore_index=True)

print("\nVerdict value counts:")
print(df["verdict"].value_counts())
print("\nCode location value counts:")
print(df["code_location"].value_counts())


In [ ]:
popularity_column = "app_popularity" if "app_popularity" in df.columns else "apk_category"

category_order = [
    "<100", "100-500", "500-1k", "1k-5k", "5k-10k", "10k-50k",
    "50k-100k", "100k-500k", "500k-1M", "1M-5M", ">5M",
]
ord_map = {category: index for index, category in enumerate(category_order)}

df["app_popularity_encoded"] = df[popularity_column].map(ord_map)
df = df.dropna(subset=["app_popularity_encoded", "apk_size"]).copy()
df["app_popularity_encoded"] = df["app_popularity_encoded"].astype(int)

scaler = RobustScaler()
df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])

print("Popularity column:", popularity_column)
print("\nApp popularity encoded value counts:")
print(df["app_popularity_encoded"].value_counts().sort_index())
print("\napk_size_scaled summary:")
print(df["apk_size_scaled"].describe())


In [ ]:
# Statistical test helpers are shared in utils/statistical_tests.py.


In [ ]:
def all_correlation_tests():
    tests.kruskal_test_continuous_by_groups(df, "app_popularity_encoded", "apk_size_scaled")
    tests.logistic_regression_numeric_to_binary(df, "app_popularity_encoded", "verdict")
    tests.chi_square_test_cat_to_binary(df, "app_popularity_encoded", "verdict")
    tests.logistic_regression_numeric_to_binary(df, "app_popularity_encoded", "combined")
    tests.chi_square_test_cat_to_binary(df, "app_popularity_encoded", "combined")
    tests.logistic_regression_binary_to_binary(df, "combined", "verdict")
    tests.logistic_regression_numeric_to_binary(df, "apk_size_scaled", "combined")
    tests.mann_whitney_test(df, "apk_size_scaled", "combined")
    tests.logistic_regression_numeric_to_binary(df, "apk_size_scaled", "verdict")
    tests.mann_whitney_test(df, "apk_size_scaled", "verdict")
    tests.mann_whitney_test(df, "app_popularity_encoded", "combined")
    tests.correlation_test_treatment_to_outcome(df)


all_correlation_tests()


In [ ]:
causal_graph = """
digraph {
    combined -> verdict;
    apk_size_scaled -> combined;
    apk_size_scaled -> verdict;
    app_popularity_encoded -> combined;
    app_popularity_encoded -> verdict;
    app_popularity_encoded -> apk_size_scaled;
}
"""

display(graphviz.Source(causal_graph))

model = CausalModel(
    data=df,
    treatment="combined",
    outcome="verdict",
    graph=causal_graph.replace("\n", " "),
    common_causes=["apk_size_scaled", "app_popularity_encoded"],
)

identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print("\n------- Identified estimand -------")
print(identified_estimand)


In [ ]:
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
    target_units="ate",
    test_significance=True,
    confidence_intervals="bootstrap",
    method_params={
        "num_simulations": 300,
        "sample_size_fraction": 1.0,
        "confidence_level": 0.95,
    },
)

print("\n------- Causal effect estimate -------")
print(estimate)


In [ ]:
estimate.interpret(method_name="textual_effect_interpreter")

refutation_methods = [
    "random_common_cause",
    "placebo_treatment_refuter",
    "data_subset_refuter",
]

for method in refutation_methods:
    if method == "placebo_treatment_refuter":
        ref = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name=method,
            placebo_type="permute",
            num_simulations=200,
        )
    elif method == "data_subset_refuter":
        ref = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name=method,
            subset_fraction=0.8,
            num_simulations=200,
        )
    else:
        ref = model.refute_estimate(identified_estimand, estimate, method_name=method)

    print(f"\nRefuter: {method}\n", ref)
